# HiProbCBM - Analisis kualitatif penjelasan multilevel dan grafik naskah

Notebook ini **tidak melatih** apa pun. Ia membutuhkan hasil notebook 01-10 di Drive.

1. **Grafik kuantitatif** dari `intervention_curves.json`, `uncertainty_analysis.json`, dan `discovered_subconcepts.json` (mean +/- std tiga seed).
2. **Penjelasan multilevel** (Bab IV.7.2): untuk model utama seed 42 pada CUB dan PseudoKitchens, lima sampel uji (2 benar paling yakin, 1 benar paling ragu, 2 salah paling yakin). Untuk tiap sampel: konsep induk dengan kontribusi terbesar ke kelas prediksi, subkonsepnya dengan bobot learned attention, probabilitas dan uncertainty pada setiap tingkat, serta citra prototipe subkonsep dari data train.

Semua gambar dan JSON disimpan di `HiProbCBM_results/figures/` di Drive. Jalankan semua sel dari atas ke bawah pada runtime GPU T4 (VPN dimatikan).


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import torch

DRIVE_ROOT = Path("/content/drive/MyDrive/TESIS AZHIM")
assert DRIVE_ROOT.is_dir(), f"Folder tesis tidak ditemukan: {DRIVE_ROOT}"
print("Drive siap:", DRIVE_ROOT)


## 2. Clone/perbarui repo dan instal dependensi (termasuk CLIP)

In [ ]:
%cd /content
!test -d HiProbCBM || git clone https://github.com/Azhimanich/HiProbCBM.git
%cd /content/HiProbCBM
!git pull --ff-only
!pip install -e ".[clip]" -q


## 3. Siapkan CUB dan PseudoKitchens dari Drive

In [ ]:
import os
import pickle

DRIVE_CUB_ROOT = DRIVE_ROOT / "CUB_200_2011"
assert (DRIVE_CUB_ROOT / "images").is_dir(), f"Folder CUB tidak ditemukan: {DRIVE_CUB_ROOT}"
metadata = DRIVE_CUB_ROOT / "metadata"
needed = [metadata / name for name in ("train.pkl", "val.pkl", "test.pkl")]
assert all(path.is_file() for path in needed), "metadata/train.pkl, val.pkl, test.pkl harus ada di Drive."

# Normalisasi ini idempotent: hanya menulis ulang pkl bila masih ada prefix lama.
marker = "CUB_200_2011/images/"
for path in needed:
    with path.open("rb") as handle:
        records = pickle.load(handle)
    changed = sum(marker in record["img_path"] for record in records)
    if changed:
        for record in records:
            if marker in record["img_path"]:
                record["img_path"] = record["img_path"].split(marker, 1)[1]
        with path.open("wb") as handle:
            pickle.dump(records, handle)
        print(f"Dinormalisasi: {path.name} ({changed} path)")
    else:
        print(f"OK: {path.name} ({len(records)} record)")

WORK_ROOT = Path("/content/HiProbCBM/datasets/CUB_200_2011")
WORK_ROOT.parent.mkdir(parents=True, exist_ok=True)
if WORK_ROOT.exists() or WORK_ROOT.is_symlink():
    assert WORK_ROOT.is_symlink() and WORK_ROOT.resolve() == DRIVE_CUB_ROOT.resolve(), (
        f"Symlink CUB tidak sesuai: {WORK_ROOT}. Jangan hapus folder non-symlink secara otomatis."
    )
else:
    os.symlink(DRIVE_CUB_ROOT, WORK_ROOT, target_is_directory=True)
print("CUB siap:", WORK_ROOT, "->", DRIVE_CUB_ROOT)


In [ ]:
import os

DRIVE_KITCHENS_ROOT = DRIVE_ROOT / "PseudoKitchens_processed"
assert (DRIVE_KITCHENS_ROOT / "manifest.csv").is_file(), (
    f"manifest.csv tidak ditemukan: {DRIVE_KITCHENS_ROOT}. Jalankan notebook setup/konversi PseudoKitchens dahulu."
)
assert (DRIVE_KITCHENS_ROOT / "info.json").is_file(), "info.json PseudoKitchens tidak ditemukan."

WORK_ROOT = Path("/content/HiProbCBM/datasets/PseudoKitchens")
WORK_ROOT.parent.mkdir(parents=True, exist_ok=True)
if WORK_ROOT.exists() or WORK_ROOT.is_symlink():
    assert WORK_ROOT.is_symlink() and WORK_ROOT.resolve() == DRIVE_KITCHENS_ROOT.resolve(), (
        f"Symlink PseudoKitchens tidak sesuai: {WORK_ROOT}. Jangan hapus folder non-symlink secara otomatis."
    )
else:
    os.symlink(DRIVE_KITCHENS_ROOT, WORK_ROOT, target_is_directory=True)
print("PseudoKitchens siap:", WORK_ROOT, "->", DRIVE_KITCHENS_ROOT)


## 4. Sambungkan train_log ke Drive dan cek GPU

In [ ]:
TRAIN_LOG_DRIVE = DRIVE_ROOT / "HiProbCBM_results" / "train_log"
TRAIN_LOG_DRIVE.mkdir(parents=True, exist_ok=True)
TRAIN_LOG_LOCAL = Path("/content/HiProbCBM/train_log")
if TRAIN_LOG_LOCAL.exists() or TRAIN_LOG_LOCAL.is_symlink():
    assert TRAIN_LOG_LOCAL.is_symlink() and TRAIN_LOG_LOCAL.resolve() == TRAIN_LOG_DRIVE.resolve(), (
        f"train_log lokal bukan symlink yang diharapkan: {TRAIN_LOG_LOCAL}"
    )
else:
    os.symlink(TRAIN_LOG_DRIVE, TRAIN_LOG_LOCAL, target_is_directory=True)

def assert_persistent_training_storage():
    assert Path("/content/drive/MyDrive").is_dir(), "Google Drive belum mounted."
    assert TRAIN_LOG_LOCAL.is_symlink(), "train_log harus symlink ke Drive."
    assert TRAIN_LOG_LOCAL.resolve() == TRAIN_LOG_DRIVE.resolve(), "Target train_log berubah."

assert_persistent_training_storage()
assert torch.cuda.is_available(), "GPU belum aktif. Pilih runtime GPU sebelum training."
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint Drive:", TRAIN_LOG_DRIVE)


## 5. Persiapan

In [ ]:
import json
from IPython.display import Image as ShowImage, display
from hiprobcbm.config import Config, load_config, prepare_log_dir, resolve_device
from hiprobcbm import reporting
from hiprobcbm.utils.seed import set_random_seed

FIG_DIR = DRIVE_ROOT / "HiProbCBM_results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SEEDS = [42, 43, 44]
# varian -> (config Tahap 1, config Tahap 2) per protokol
PROTOKOL_CONFIG = {
    "probcbm": {"utama": ("configs/cub_stage1_probcbm_reported.yaml", "configs/cub_stage2_probcbm_reported.yaml"),
                "a1": ("configs/cub_stage1_probcbm_reported.yaml", "configs/cub_stage2_probcbm_reported.yaml"),
                "a2": ("configs/cub_stage1_probcbm_reported.yaml", "configs/cub_stage2_probcbm_reported.yaml"),
                "a3": ("configs/cub_stage1_probcbm_reported_batchtopk.yaml", "configs/cub_stage2_probcbm_reported_batchtopk.yaml")},
    "hicem": {"utama": ("configs/kitchens_stage1_hicem_reported.yaml", "configs/kitchens_stage2_hicem_reported.yaml"),
              "a1": ("configs/kitchens_stage1_hicem_reported.yaml", "configs/kitchens_stage2_hicem_reported.yaml"),
              "a2": ("configs/kitchens_stage1_hicem_reported.yaml", "configs/kitchens_stage2_hicem_reported.yaml"),
              "a3": ("configs/kitchens_stage1_hicem_reported_batchtopk.yaml", "configs/kitchens_stage2_hicem_reported_batchtopk.yaml")},
}
DATASET_NAME = {"probcbm": "CUB-200-2011 / ResNet18", "hicem": "PseudoKitchens / CLIP ViT-L/14"}
FILE_TAG = {"probcbm": "cub", "hicem": "pseudokitchens"}

def run_dirs(protokol, varian, seed):
    stage1_config, stage2_config = PROTOKOL_CONFIG[protokol][varian]
    induk = "a3" if varian == "a3" else "utama"  # A1/A2 memakai hierarchy utama
    parent = prepare_log_dir(load_config(stage1_config), f"train_log/{protokol}/{induk}/seed_{seed}")
    values = load_config(stage2_config).to_dict()
    if varian in {"a1", "a2"}:
        values["experiment_name"] = f"{values['experiment_name']}_ablation_{varian}"
    child = prepare_log_dir(Config(values), f"train_log/{protokol}/{varian}/seed_{seed}")
    return parent, child

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

assert_persistent_training_storage()
print("Gambar disimpan di:", FIG_DIR)


## 6. Grafik kuantitatif

In [ ]:
for protokol in ("probcbm", "hicem"):
    curves, groups, discovered = {}, {}, {}
    for varian in ("utama", "a1", "a2", "a3"):
        for seed in SEEDS:
            _, child = run_dirs(protokol, varian, seed)
            c = read_json(child / "intervention_curves.json")
            u = read_json(child / "uncertainty_analysis.json")
            d = read_json(child / "discovered_subconcepts.json")
            if c: curves.setdefault(varian, []).append(c)
            if u: groups.setdefault(varian, []).append(u)
            if d and varian in ("utama", "a3"): discovered.setdefault(varian, []).append((seed, d))
        print(protokol, varian, "| kurva intervensi:", len(curves.get(varian, [])),
              "| analisis uncertainty:", len(groups.get(varian, [])), "seed")
    tag, name = FILE_TAG[protokol], DATASET_NAME[protokol]
    outputs = [reporting.plot_intervention_curves(curves, FIG_DIR / f"intervensi_konsep_{tag}.png",
                                                  f"Intervensi konsep induk - {name}"),
               reporting.plot_uncertainty_groups(groups, FIG_DIR / f"uncertainty_vs_akurasi_{tag}.png",
                                                 f"Uncertainty vs akurasi (10 kelompok) - {name}")]
    if discovered:
        outputs.append(reporting.plot_discovered_interventions(
            discovered, FIG_DIR / f"intervensi_subkonsep_{tag}.png", f"Intervensi subkonsep hasil discovery - {name}"))
    for path in outputs:
        print(path)
        display(ShowImage(filename=str(path)))


## 7. Penjelasan multilevel per sampel

In [ ]:
# Penjelasan multilevel model utama, seed 42: 2 sampel benar paling yakin,
# 1 sampel benar paling ragu, 2 sampel salah paling yakin.
device = resolve_device(0)
SEED = 42
for protokol in ("probcbm", "hicem"):
    parent, child = run_dirs(protokol, "utama", SEED)
    cfg = Config(json.loads((child / "run_manifest.json").read_text(encoding="utf-8"))["config"])
    set_random_seed(cfg.get("seed", SEED))
    names = reporting.sub_names_from_discovered(child / "discovered_subconcepts.json")
    result = reporting.explain_test_samples(cfg, child / "stage2_best.pth", parent / "pseudo_hierarchy.pt", device,
                                            sub_names=names)
    tag = FILE_TAG[protokol]
    (FIG_DIR / f"penjelasan_multilevel_{tag}.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    for n, sample in enumerate(result["samples"], 1):
        path = reporting.plot_explanation(sample, FIG_DIR / f"penjelasan_multilevel_{tag}_{n}.png",
                                          f"Penjelasan multilevel HiProbCBM - {DATASET_NAME[protokol]} (sampel uji {sample['test_index']})")
        print(path)
        display(ShowImage(filename=str(path)))
